In [4]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import chi2

# ---------------------------
# Data
# ---------------------------
control_dist = [0.6484, 0.6310, 0.6835, 0.3532, 0.4503, 0.4678,
                0.4226, 0.4540, 0.5091, 0.4670, 0.5697, 0.5820]
control_animal = ['G33', 'G33', 'G33', 'G34', 'G34', 'G34',
                  'G35', 'G35', 'G35', 'G36', 'G36', 'G36']

stim_dist = [0.8543, 0.8024, 0.8918, 0.7506, 0.8326, 0.8286,
             0.7508, 0.7120, 0.8152, 0.7978,
             0.5823, 0.7944, 0.6782,
             0.4742, 0.3286, 0.4824,
             0.7652, 0.7087, 0.6085, 0.6523]
stim_animal = ['G7', 'G7',
               'G8', 'G8', 'G8', 'G8',
               'G9', 'G9', 'G9', 'G9',
               'G28', 'G28', 'G28',
               'G29', 'G29', 'G29',
               'G37', 'G37', 'G37', 'G37']

stress_dist = [1.001113156, 1.002419683, 0.969890523,
               0.7640226, 0.790428686,
               0.885770891, 0.810616632]
stress_animal = ['G7', 'G8', 'G9', 'G28', 'G29', 'G37', 'G37']


# ---------------------------
# Build DataFrame
# ---------------------------
def build_df(animal, values, condition):
    return pd.DataFrame({
        'Animal_id': animal,
        'y': values,
        'condition': condition
    })

df = pd.concat([
    build_df(control_animal, control_dist, 'control'),
    build_df(stim_animal, stim_dist, 'stim'),
    build_df(stress_animal, stress_dist, 'stress')
], ignore_index=True)

df['condition'] = pd.Categorical(
    df['condition'],
    categories=['control', 'stim', 'stress'],
    ordered=True
)

# ---------------------------
# Mixed model: y ~ condition + (1 | Animal_id)
# ---------------------------
model = smf.mixedlm(
    "y ~ C(condition, Treatment(reference='control'))",
    data=df,
    groups=df["Animal_id"]
)
res = model.fit(reml=True, method="lbfgs")

print(res.summary())

# Residual SD (useful for standardizing model-based contrasts)
sigma_resid = float(np.sqrt(res.scale))

# ---------------------------
# Helper: raw Cohen's d using pooled SD (ignores clustering/pairing)
# ---------------------------
def cohens_d_pooled(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    sx, sy = np.var(x, ddof=1), np.var(y, ddof=1)
    sp = np.sqrt(((nx - 1) * sx + (ny - 1) * sy) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / sp

# ---------------------------
# Post-hoc contrasts + effect sizes
# ---------------------------
fe_names = res.fe_params.index.tolist()
name_stim = "C(condition, Treatment(reference='control'))[T.stim]"
name_stress = "C(condition, Treatment(reference='control'))[T.stress]"

def weight_vector(**kwargs):
    v = np.zeros(len(fe_names))
    for k, val in kwargs.items():
        v[fe_names.index(k)] = val
    return v

def run_contrast(weights, label, groupA, groupB):
    # Mixed-model contrast
    t = res.t_test(np.asarray([weights]))
    est = float(t.effect)
    se = float(t.sd)
    tval = float(t.tvalue)
    p = float(t.pvalue)

    # Model-standardized effect size
    d_model = est / sigma_resid

    # Raw Cohen's d (pooled SD; quick descriptive ES)
    x = df.loc[df['condition'] == groupA, 'y'].values
    y = df.loc[df['condition'] == groupB, 'y'].values
    d_raw = cohens_d_pooled(x, y)

    return {
        "contrast": label,
        "estimate": est,
        "SE": se,
        "t": tval,
        "p_raw": p,
        "d_model(Δ/σ_resid)": d_model,
        "d_raw_pooled": d_raw,
        "sigma_resid": sigma_resid
    }

posthoc = pd.DataFrame([
    run_contrast(weight_vector(**{name_stim: 1}), "stim − control", "stim", "control"),
    run_contrast(weight_vector(**{name_stress: 1}), "stress − control", "stress", "control"),
    run_contrast(weight_vector(**{name_stress: 1, name_stim: -1}), "stress − stim", "stress", "stim")
])

# Holm correction
rej, p_adj, _, _ = multipletests(posthoc["p_raw"], method="holm")
posthoc["p_adj_holm"] = p_adj
posthoc["reject_0.05"] = rej
posthoc["CIU"] = posthoc["estimate"] + 1.96 * posthoc["SE"]
posthoc["CIL"] = posthoc["estimate"] - 1.96 * posthoc["SE"]
posthoc["reject_0.05"] = rej
print("\nPost-hoc contrasts with effect sizes (Holm-corrected):")
print(posthoc[[
    "contrast", "estimate", "SE", "t", "p_raw", "p_adj_holm", "reject_0.05",
    "d_model(Δ/σ_resid)", "d_raw_pooled", "CIL", "CIU"
]])



                             Mixed Linear Model Regression Results
Model:                          MixedLM               Dependent Variable:               y      
No. Observations:               39                    Method:                           REML   
No. Groups:                     10                    Scale:                            0.0045 
Min. group size:                3                     Log-Likelihood:                   32.3170
Max. group size:                6                     Converged:                        Yes    
Mean group size:                3.9                                                            
-----------------------------------------------------------------------------------------------
                                                       Coef. Std.Err.   z   P>|z| [0.025 0.975]
-----------------------------------------------------------------------------------------------
Intercept                                              0.520    0.062

/var/folders/3_/93qzqh6s34ldm6zlq4mw0hs40000gp/T/ipykernel_74090/71673247.py:96: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  est = float(t.effect)
/var/folders/3_/93qzqh6s34ldm6zlq4mw0hs40000gp/T/ipykernel_74090/71673247.py:97: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  se = float(t.sd)
/var/folders/3_/93qzqh6s34ldm6zlq4mw0hs40000gp/T/ipykernel_74090/71673247.py:98: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  tval = float(t.tvalue)
